In [0]:
%pip install polars pandas scikit-learn matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.2/865.2 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.5/52.5 MB 60.0 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# COMMAND ----------

# MAGIC %md
# MAGIC # 1. Data Loading & Setup
# MAGIC Data Validation: Remove invalid judge-case assignments

# COMMAND ----------

%pip install polars pandas scikit-learn matplotlib seaborn

# COMMAND ----------

import polars as pl
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print(" Libraries imported")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Load Reference Data

# COMMAND ----------

judges = pl.read_csv("/Workspace/Users/sumedha190198@gmail.com/judicial-backlog/judges_clean.csv")
disp_key = pl.read_csv("/Workspace/Users/sumedha190198@gmail.com/judicial-backlog/disp_name_key.csv")
type_key = pl.read_csv("/Workspace/Users/sumedha190198@gmail.com/judicial-backlog/type_name_key.csv")
purpose_key = pl.read_csv("/Workspace/Users/sumedha190198@gmail.com/judicial-backlog/purpose_name_key.csv")

print(f" Judges: {len(judges):,}")
print(f" Dispositions: {len(disp_key):,}")
print(f" Types: {len(type_key):,}")
print(f" Purposes: {len(purpose_key):,}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Load Cases Data (Lazy)

# COMMAND ----------

lf = pl.scan_parquet("/Workspace/Users/sumedha190198@gmail.com/judicial-backlog/cases_sample_10pct.parquet")

print(f"Cases loaded")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Prepare Judges 

# COMMAND ----------

judges = judges.with_columns([
    pl.col("start_date").str.to_date("%d-%m-%Y", strict=False).alias("judge_start"),
    pl.col("end_date").str.to_date("%d-%m-%Y", strict=False).alias("judge_end"),
])

# Remove judges with invalid dates
judges_valid = judges.filter(pl.col("judge_start").is_not_null())

print(f"Judges before validation: {len(judges):,}")
print(f"Judges after validation: {len(judges_valid):,}")
print(f"Removed (invalid dates): {len(judges) - len(judges_valid):,}")

judges_lazy = judges_valid.lazy()

print(f" Judges prepared with validation")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Filter Valid Cases 

# COMMAND ----------

# Step 1: Filter by duration
lf_filtered = (
    lf
    .filter((pl.col("duration_days") >= 0) & (pl.col("duration_days") <= 7300))
)

print(f" Duration filter applied: 0-7300 days")

# Step 2: Join with judges and validate tenure
lf_with_judges = (
    lf_filtered
    .join(judges_lazy.select(["state_code", "dist_code", "court_no", "judge_position", "judge_start"]),
          on=["state_code", "dist_code", "court_no", "judge_position"], how="left")
)

# Step 3: Calculate tenure and remove invalid values
lf_valid = (
    lf_with_judges
    .with_columns([
        (pl.col("filing_date").dt.year() - pl.col("judge_start").dt.year()).alias("judge_tenure_years")
    ])
    # CRITICAL FILTER: Only cases where judge_start is NOT NULL and filing_date >= judge_start
    .filter(pl.col("judge_start").is_not_null())
    .filter(pl.col("filing_date") >= pl.col("judge_start"))
    .filter(pl.col("judge_tenure_years") >= 0)  # Non-negative tenure
    .filter(pl.col("judge_tenure_years") <= 50)  # Realistic tenure (max 50 years)
)

print(f" Tenure validation complete:")
print(f"  - Removed cases with invalid judge assignments")
print(f"  - Removed negative tenure values")
print(f"  - Valid tenure range: 0-50 years")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Summary

# COMMAND ----------

# Collect basic stats
stats = lf_valid.select([
    pl.len().alias("total_cases"),
    pl.col("duration_days").mean().alias("avg_duration"),
    pl.col("judge_tenure_years").mean().alias("avg_tenure"),
]).collect().to_dicts()[0]

print(f"""
DATA LOADING COMPLETE (VALIDATED)
==========================================
 Judges: {len(judges_valid):,} (validated)
 Reference Keys: All loaded
 Cases Sample: Loaded and filtered
 Total valid cases: {stats['total_cases']:,}
 Average duration: {stats['avg_duration']:.0f} days
 Average tenure: {stats['avg_tenure']:.1f} years

""")

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
✓ Libraries imported
✓ Judges: 98,478
✓ Dispositions: 462
✓ Types: 62,714
✓ Purposes: 68,125
Cases loaded
Judges before validation: 98,478
Judges after validation: 98,478
Removed (invalid dates): 0
✓ Judges prepared with validation
✓ Duration filter applied: 0-7300 days
✓ Tenure validation complete:
  - Removed cases with invalid judge assignments
  - Removed negative tenure values
  - Valid tenure range: 0-50 years

DATA LOADING COMPLETE (VALIDATED)
 Judges: 98,478 (validated)
 Reference Keys: All loaded
 Cases Sample: Loaded and filtered
 Total valid cases: 5,333,314
 Average duration: 277 days
 Average tenure: 3.2 years


